# Project Overview
This project analyzes the European Data Science job market for 2024 using a Kaggle dataset. The main goals are:
- Explore trends in data science jobs across Europe
- Understand skills demand, salaries, and job types
- Visualize findings interactively for a portfolio dashboard

## Dataset
- File: job_postings_monthly.xlsx
- Sheets: 12 monthly sheets (Jan-Dec)
- Columns: 17, including job title, location, salary, skills, and work type
- Rows: 478,895

## Objectives for this Notebook
1. Load and combine all monthly sheets
2. Explore the dataset (EDA)
3. Clean and preprocess for analysis
4. Prepare for visualization and dashboard integration

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# all_sheets = pd.read_excel("../data/raw/job_postings_monthly.xlsx", sheet_name=None)
df_clean = pd.read_csv("../data/processed/df_clean.csv")


In [ ]:
# Check sheet names
print("Sheets available:", list(all_sheets.keys()))

# Combine all monthly sheets into a single DataFrame
df_all = pd.concat(all_sheets.values(), ignore_index=True)

# Quick check of combined data
df_all.shape  # Total rows and columns
df_all.head(10) # First 10 rows

Sheets available: ['01-2024', '02-2024', '03-2024', '04-2024', '05-2024', '06-2024', '07-2024', '08-2024', '09-2024', '10-2024', '11-2024', '12-2024']


,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
0,Data Analyst,"Summer Internship -Data Analyst Intern, Risk M...","Marlborough, MA",via Boatingrevealed.com,"Full-time, Part-time, and Internship",False,"New York, United States",2024-01-01 00:00:01,False,True,United States,NaN,NaN,NaN,BJ's Wholesale Club,['excel'],{'analyst_tools': ['excel']}
1,Data Analyst,"Staff Data Analyst Operations, Infrastructure ...","Fremont, CA",via ClimateTechList,Full-time,False,"California, United States",2024-01-01 00:00:11,True,False,United States,NaN,NaN,NaN,Tesla,"['tableau', 'flow']","{'analyst_tools': ['tableau'], 'other': ['flow']}"
2,Data Analyst,Junior Data Analyst - Entry Level,"Waco, TX",via ZipRecruiter,Full-time and Part-time,False,"Texas, United States",2024-01-01 00:00:15,True,False,United States,NaN,NaN,NaN,Next Recruiting,NaN,NaN
3,Data Analyst,"Data Analyst/Engineer, Supply Chain Optimizati...","Austin, TX",via ClimateTechList,Internship,False,"Texas, United States",2024-01-01 00:00:18,False,False,United States,NaN,NaN,NaN,Tesla,['spring'],{'libraries': ['spring']}
4,Data Scientist,It analyst,"Tampa, FL",via Talent.com,Full-time,False,"Florida, United States",2024-01-01 00:00:29,True,False,United States,NaN,NaN,NaN,VirtualVocations,NaN,NaN
5,Senior Data Scientist,"Senior Data Scientist, Growth",Anywhere,via ZipRecruiter,Full-time,True,"California, United States",2024-01-01 00:00:43,False,True,United States,NaN,NaN,NaN,Atlassian,"['sql', 'python', 'r', 'c', 'tableau', 'micros...","{'analyst_tools': ['tableau', 'microstrategy',..."
6,Machine Learning Engineer,"Machine Learning Scientist, Prescient Design","San Ramon, CA",via Monster,Full-time,False,"California, United States",2024-01-01 00:00:46,False,False,United States,NaN,NaN,NaN,Genentech,"['python', 'pytorch', 'github']","{'libraries': ['pytorch'], 'other': ['github']..."
7,Machine Learning Engineer,Associate Machine Learning Scientist - MLDD,"Menlo Park, CA",via Monster,Full-time,False,"California, United States",2024-01-01 00:00:46,False,False,United States,NaN,NaN,NaN,Genentech,"['python', 'pytorch', 'tensorflow', 'github']","{'libraries': ['pytorch', 'tensorflow'], 'othe..."
8,Data Analyst,Data analyst,"Dallas, TX",via Talent.com,Full-time,False,"Texas, United States",2024-01-01 00:00:53,True,False,United States,NaN,NaN,NaN,TEKsystems,NaN,NaN
9,Data Scientist,"Associate Director, Data Science and Analytics","Birmingham, AL",via BeBee,Full-time,False,"Illinois, United States",2024-01-01 00:01:00,False,True,United States,NaN,NaN,NaN,Razorfish,"['sql', 'r', 'python', 'tableau', 'power bi']","{'analyst_tools': ['tableau', 'power bi'], 'pr..."


In [5]:
df_all.shape
df_all.columns
df_all.info()
df_all.isnull().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478895 entries, 0 to 478894
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   job_title_short        478895 non-null  object        
 1   job_title              478894 non-null  object        
 2   job_location           477522 non-null  object        
 3   job_via                478889 non-null  object        
 4   job_schedule_type      470983 non-null  object        
 5   job_work_from_home     478895 non-null  bool          
 6   search_location        478895 non-null  object        
 7   job_posted_date        478895 non-null  datetime64[ns]
 8   job_no_degree_mention  478895 non-null  bool          
 9   job_health_insurance   478895 non-null  bool          
 10  job_country            478574 non-null  object        
 11  salary_rate            30411 non-null   object        
 12  salary_year_avg        20335 non-null   floa

salary_hour_avg          468819
salary_year_avg          458560
salary_rate              448484
job_type_skills           68870
job_skills                68870
job_schedule_type          7912
job_location               1373
job_country                 321
company_name                  8
job_via                       6
job_title                     1
job_title_short               0
job_work_from_home            0
job_no_degree_mention         0
job_posted_date               0
search_location               0
job_health_insurance          0
dtype: int64

In [14]:
# ==============================
# DATA CLEANING - CREATE df_clean
# ==============================

# Create a copy of the combined dataset
df_clean = df_all.copy()

# ------------------------------
# 1. Keep only relevant columns
# ------------------------------
columns_to_keep = [
    'job_title_short',
    'job_title',
    'job_location',
    'job_country',
    'job_work_from_home',
    'job_posted_date',
    'job_no_degree_mention',
    'salary_year_avg',
    'job_skills',
    'job_type_skills',
    'job_posted_date'
]

df_clean = df_clean[columns_to_keep]

# ------------------------------
# 2. Handle missing countries
# ------------------------------
# Fill missing job_country using job_location
df_clean['job_country'] = df_clean['job_country'].fillna(
    df_clean['job_location'].apply(lambda x: str(x).split(',')[-1].strip())
)

# ------------------------------
# 3. Remove critical missing rows
# ------------------------------
# Only drop rows where BOTH job title and country are missing
df_clean = df_clean.dropna(subset=['job_title_short', 'job_country'])

# ------------------------------
# 4. Clean text fields
# ------------------------------
# Lowercase skills for consistency
df_clean['job_skills'] = df_clean['job_skills'].str.lower()

# ------------------------------
# 5. Reset index after cleaning
# ------------------------------
df_clean = df_clean.reset_index(drop=True)

# ------------------------------
# 6. Extract experience level from job_title
# ------------------------------
def extract_experience(title):
    if pd.isna(title):
        return 'Not Specified'
    
    title = str(title).lower()

    # Entry level
    if any(keyword in title for keyword in [
        'intern', 'internship', 'junior', 'jr', 'graduate', 'entry', 'associate'
    ]):
        return 'Entry'
    
    # Mid level
    elif any(keyword in title for keyword in [
        'mid', 'mid-level'
    ]):
        return 'Mid'
    
    # Senior level
    elif any(keyword in title for keyword in [
        'senior', 'sr', 'staff', 'expert'
    ]):
        return 'Senior'

    # Lead level
    elif any(keyword in title for keyword in [
        'lead', 'principal', 'manager', 'head'
    ]):
        return 'Lead'

    # Default fallback
    else:
        return 'Not Specified'

# Apply function
df_clean['experience_level'] = df_clean['job_title'].apply(extract_experience)

df_clean['experience_level'].value_counts()

# Save cleaned dataset to processed folder
df_clean.to_csv("../data/processed/df_clean.csv", index=False)

In [ ]:
# ------------------------------
# 7. Clean job_title_short (remove experience words)
# ------------------------------
def clean_job_title(title):
    if pd.isna(title):
        return title
    
    title = str(title).lower()

    # Words to remove
    remove_words = [
        'senior', 'sr', 'sr.', 
        'junior', 'jr', 'jr.', 
        'lead', 'principal', 
        'staff', 'manager', 
        'intern', 'internship', 
        'associate', 'expert', 
        'graduate'
    ]

    # Remove unwanted words
    for word in remove_words:
        title = title.replace(word, '')

    # Clean extra spaces
    title = " ".join(title.split())

    # Capitalize nicely
    return title.title()

df_clean['job_title_clean'] = df_clean['job_title_short'].apply(clean_job_title)
df_clean[['job_title_short', 'job_title_clean']].drop_duplicates().head(20)

df_clean['job_title_short'] = df_clean['job_title_clean']
df_clean = df_clean.drop(columns=['job_title_clean'])

df_clean['job_title_short'].value_counts()

job_title_short
Data Engineer                159602
Data Analyst                 128213
Data Scientist               119395
Business Analyst              28584
Software Engineer             23402
Machine Learning Engineer     12860
Cloud Engineer                 6839
Name: count, dtype: int64

In [ ]:
# ------------------------------
# 8. Separating skills in a different file
# ------------------------------
import ast

df_skills = df_clean.copy()
df_skills = df_skills[
    [
        'job_title_short',
        'job_country',
        'experience_level',
        'job_type_skills',
        'job_no_degree_mention'
    ]
]
# Remove rows without skills
df_skills = df_skills.dropna(subset=['job_type_skills'])

# Convert string -> dictionary
df_skills['job_type_skills'] = df_skills['job_type_skills'].apply(ast.literal_eval)

rows = []

for _, row in df_skills.iterrows():
    skills_dict = row['job_type_skills']
    
    for skill_type, skills_list in skills_dict.items():
        for skill in skills_list:
            rows.append({
                'job_title_short': row['job_title_short'],
                'job_country': row['job_country'],
                'experience_level': row['experience_level'],
                'skill': skill.lower(),
                'skill_type': skill_type
            })

df_skills_final = pd.DataFrame(rows)
df_skills_final.head()
df_skills_final.shape
df_skills_final.to_csv("../data/processed/df_skills.csv", index=False)